[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/43_flow_matching_solution.ipynb)

# Solution: Conditional Flow Matching

A complete 2D Conditional Flow Matching example.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


def sample_p0(batch_size: int, device=None):
    return torch.randn(batch_size, 2, device=device)


def sample_p1(batch_size: int, device=None):
    t = torch.rand(batch_size, device=device) * (2 * torch.pi)
    r = 2.0 + 0.1 * torch.randn(batch_size, device=device)
    return torch.stack((r * torch.cos(t), r * torch.sin(t) * torch.cos(t)), dim=1)


class VectorFieldNet(nn.Module):
    def __init__(self, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, 2))

    def forward(self, x, t):
        if t.ndim == 1:
            t = t.unsqueeze(1)
        return self.net(torch.cat((x, t), dim=1))


def compute_fm_loss(model, x0, x1):
    t = torch.rand(x0.shape[0], 1, device=x0.device, dtype=x0.dtype)
    x_t = (1.0 - t) * x0 + t * x1
    target_v = x1 - x0
    return ((model(x_t, t) - target_v) ** 2).mean()


@torch.no_grad()
def sample_ode(model, x0, steps=50):
    was_training = model.training
    model.eval()
    x = x0.clone()
    dt = 1.0 / steps
    for step in range(steps):
        t = torch.full((x.shape[0], 1), step * dt, device=x.device, dtype=x.dtype)
        x = x + dt * model(x, t)
    model.train(was_training)
    return x


In [ ]:
# Optional full training run
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VectorFieldNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
for _ in range(2000):
    x0, x1 = sample_p0(512, device), sample_p1(512, device)
    loss = compute_fm_loss(model, x0, x1)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

generated = sample_ode(model, sample_p0(1_000, device), steps=100).cpu()
target = sample_p1(1_000).cpu()
plt.scatter(target[:, 0], target[:, 1], s=4, alpha=0.5, label='target')
plt.scatter(generated[:, 0], generated[:, 1], s=4, alpha=0.5, label='generated')
plt.axis('equal'); plt.legend(); plt.show()


In [ ]:
from torch_judge import check
check('flow_matching')
